# Cuaderno 3 · Cuando tus datos se acaban

**Descripción y Visualización de Datos · UAI 2026 · Clase 3**

La clase pasada aprendimos a **describir** una variable a la vez: el promedio de edad, la tabla de
comunas, el histograma de estaturas.

Hoy vamos a hacer la pregunta que de verdad importa: **¿es distinto un grupo de otro?** Para eso hay
que cruzar dos variables. Y en el camino nos vamos a topar con un muro que **no se arregla con más
código**.

Ese muro es el motivo de todo lo que viene después en el curso.

---

**Antes de empezar: `Archivo → Guardar una copia en Drive`.** Este cuaderno es del profesor. Si
escribes acá sin hacer tu copia, se pierde todo.

## Parte 1 · Volver a nuestra base

Es la misma encuesta que respondieron ustedes. Volvemos a cargarla y a ponerle nombres decentes a
las columnas, igual que la clase pasada.

In [ ]:
url <- "https://docs.google.com/spreadsheets/d/1LvQhNGX-AN3GIHlQ4NRIk1V_RVdrbLGkg42U2undqIk/export?format=csv"

respuestas <- read.csv(url)

names(respuestas) <- c("marca_temporal", "edad", "estatura", "hermanos", "comuna",
                       "minutos_viaje", "transporte", "horas_sueno", "horas_redes",
                       "sistema_operativo", "tazas_cafe", "experiencia", "dominio",
                       "op_grafico_miente", "op_interes_programar",
                       "op_datos_chile", "op_hablar_publico")

nrow(respuestas)   # cuántas personas somos

## Parte 2 · Repaso relámpago: una variable a la vez

Esto ya lo saben hacer. Una tabla para una variable de texto, un promedio para una numérica.

In [ ]:
table(respuestas$transporte)     # ¿cómo llega la gente a la universidad?

Perfecto. Pero fíjate en lo que **no** te dice esa tabla: no te dice si quienes vienen en micro
duermen distinto que quienes vienen en auto. Para eso hay que cruzar.

## Parte 3 · Cruzar dos variables

Se le pasan **dos** columnas a `table()`, separadas por coma. La primera queda en las filas, la
segunda en las columnas.

In [ ]:
tabla <- table(respuestas$transporte, respuestas$sistema_operativo)
tabla

### El problema de los números absolutos

Esa tabla tiene una trampa que ya conocen de la clase pasada: **los grupos no son del mismo
tamaño**. Hay muchas más personas que llegan en micro que en metro, así que comparar los conteos
directamente engaña.

La solución es la misma de siempre: **porcentajes dentro de cada fila**. Eso es lo que hace
`margin = 1`.

In [ ]:
round(prop.table(tabla, margin = 1) * 100)   # cada fila suma 100%

Ahora sí se puede leer: *"de los que llegan en metro, el 100% usa iPhone"*.

Guarda esa frase. **Vamos a volver a ella en un minuto, y no va a sobrevivir.**

## Parte 4 · Un número promedio según un grupo

Cuando lo que quieres comparar es un **número** (horas de sueño) entre **grupos** (medio de
transporte), la función es `tapply()`. Se lee así: *"aplica `mean` a `horas_sueno`, separando por
`transporte`"*.

In [ ]:
tapply(respuestas$horas_sueno, respuestas$transporte, mean)

### 🚨 Salió `NA` en todo. ¿Por qué?

**El código corrió. No hubo error. Y el resultado no sirve.** Ésta es la lección más importante de
la clase pasada, y acá está otra vez.

Miremos qué tiene adentro la columna:

In [ ]:
class(respuestas$horas_sueno)   # ¿qué tipo de variable es?
respuestas$horas_sueno          # miremos los valores uno por uno

Ahí está: **una sola persona escribió `"3 horas"` en vez de `3`**.

Con un solo valor de texto, R decide que **la columna entera es texto**, y el promedio de un texto
es `NA`. Una persona de 33 arruinó el cálculo para las 33.

Lo arreglamos con las mismas dos funciones de la clase pasada: `gsub()` para borrar lo que no es
número, y `as.numeric()` para convertir. **Y nunca sobre la columna original: siempre en una
columna nueva.**

In [ ]:
respuestas$horas_sueno_num <- as.numeric(gsub("[^0-9.,]", "", respuestas$horas_sueno))

round(tapply(respuestas$horas_sueno_num, respuestas$transporte, mean), 1)

Ahora sí. Y este mismo problema está en **dos columnas más** de la encuesta: alguien escribió
`"10 min"` en los minutos de viaje y alguien escribió `"5 horas"` en las horas de redes sociales.

Un gráfico para ver la comparación completa:

In [ ]:
boxplot(horas_sueno_num ~ transporte,
        data = respuestas,
        main = "Horas de sueño según medio de transporte",
        xlab = "Cómo llega a la universidad",
        ylab = "Horas que durmió anoche",
        col = "lightblue")

## Parte 5 · 🧱 El muro

Volvamos a esa frase que guardamos: *"de los que llegan en metro, el 100% usa iPhone"*.

Suena a hallazgo. Veamos sobre cuánta gente está calculado.

In [ ]:
table(respuestas$transporte)          # ¿cuántos hay en cada grupo?
table(respuestas$sistema_operativo)   # ¿cuántos en cada sistema operativo?

El "100%" del metro son **6 personas**. Y mira la categoría `Otro` del sistema operativo:
**1 persona**. Una.

Ahora saquemos los porcentajes de la misma tabla, pero **por columna** (`margin = 2`), y miremos la
columna `Otro`:

In [ ]:
round(prop.table(tabla, margin = 2) * 100)   # cada COLUMNA suma 100%

Ahí tienes un titular perfecto y completamente vacío:

> *"El **100%** de quienes usan otro sistema operativo llega a la universidad en micro."*

Ese 100% **es una persona**. Y no hay forma de que hubiera dado otra cosa: con una sola persona en
la categoría, el resultado sólo puede ser 0% o 100%. El número parece contundente justamente porque
está calculado sobre casi nada.

**Por eso un porcentaje nunca se publica sin decir sobre cuántos casos está calculado.**

### La lección de hoy

**Somos 33 personas.** Al cruzar dos variables, esos 33 se reparten en 9 o 12 casillas, y varias
quedan con 1 o 2 personas.

> **Esto no se arregla escribiendo mejor código.** No hay función en R que invente los datos que no
> tienes. La base **se acabó**.

Y ése es exactamente el punto donde empieza su proyecto del semestre: para responder preguntas
sobre un tema de verdad, hay que **salir a buscar datos que ya existen**, levantados por alguien
más, sobre miles de personas en vez de 33.

De eso se trata la segunda mitad de esta clase.

## Parte 6 · La tabla que decide sus grupos

En la encuesta, cada uno eligió el dominio de su proyecto. Veamos qué eligió el curso.

In [ ]:
sort(table(respuestas$dominio), decreasing = TRUE)

### Mírala con cuidado antes de creerle

Esa tabla tiene **10 categorías**, pero el curso no eligió 10 temas distintos. Hay tres problemas:

1. Alguien escribió **`politica`** en minúscula y sin tilde. Para R, `politica` y `Política` son
   dos cosas distintas.
2. Alguien escribió **`Medioambiente`** en una palabra.
3. Alguien **no eligió del menú** y escribió `tecnologia y salud`. ¿Ese grupo es de salud, de
   tecnología, o de los dos?

**Ninguno de los tres lo puede resolver R sola. Son decisiones tuyas, y hay que documentarlas.**

### La misma enfermedad, en otra columna

Para verlo con más fuerza, miremos la comuna donde vive cada uno:

In [ ]:
sort(table(respuestas$comuna), decreasing = TRUE)

`Las condes` y `las condes` son la misma comuna. `ñuñoa` y `Ñuñoa` también. Y Peñalolén aparece
**cuatro veces escrita de cuatro formas distintas**.

Vamos a arreglarlo por capas. **Capa 1: todo a minúscula**, con `tolower()` — una función nueva,
hace exactamente lo que su nombre dice.

In [ ]:
sort(table(tolower(respuestas$comuna)), decreasing = TRUE)

Mejoró… pero fíjate bien: **`las condes` sigue apareciendo dos veces**. ¿Por qué, si ya están las
dos en minúscula?

Porque una tiene un **espacio invisible al final**. Para R, `"las condes"` y `"las condes "` no son
lo mismo.

**Capa 2: sacar los espacios sobrantes**, con `trimws()`.

In [ ]:
comuna_limpia <- trimws(tolower(respuestas$comuna))

sort(table(comuna_limpia), decreasing = TRUE)

Casi. Todavía quedan **`ñuñoa` y `nunoa`**, y **`peñalolen` y `peñalolén`**: alguien escribió sin
tilde y sin la eñe.

Podríamos seguir limpiando, pero acá está la lección:

> **Limpiar no es un paso: son capas.** Cada vez que arreglas una, aparece otra. Y en algún momento
> tienes que **decidir dónde parar y dejarlo escrito**, para que otra persona pueda repetir
> exactamente lo que hiciste y llegar al mismo número.

## Parte 7 · El pronóstico del curso

Última pregunta de la encuesta que vamos a mirar hoy. Ustedes respondieron qué tan de acuerdo
estaban con esta afirmación:

> *"Los datos públicos en Chile son fáciles de encontrar"*

In [ ]:
table(respuestas$op_datos_chile)

**21 de 33 dijeron que están de acuerdo o muy de acuerdo.** Casi dos tercios del curso cree que
esto va a ser fácil.

Anota ese número. **Al final de la clase de hoy vamos a volver a preguntarlo**, después de ver cómo
publican sus datos el Ministerio del Deporte, la encuesta CASEN y el Servicio Electoral.

## Parte 8 · ✏️ Tu turno

Resuelve estos tres en tu copia del cuaderno.

**1.** La columna `minutos_viaje` tiene el mismo problema que tenía `horas_sueno` (alguien escribió
`"10 min"`). Créala limpia en una columna nueva llamada `minutos_num` y calcula el **promedio de
minutos de viaje según el medio de transporte**.

In [ ]:
# Tu código acá

**2.** Cruza `experiencia` (experiencia previa programando) con `sistema_operativo` y muestra los
porcentajes por fila. Después responde en una celda de texto: **¿hay alguna casilla con tan pocas
personas que el porcentaje no signifique nada?**

In [ ]:
# Tu código acá

**3.** Usando `comuna_limpia`, ¿cuál es la comuna más frecuente del curso? *(Pista: la clase pasada
usamos `names(which.max(...))`.)*

In [ ]:
# Tu código acá

## Antes de irte

**Archivo → Guardar** (Ctrl+S). Tu copia queda en tu Drive, en *Colab Notebooks*.

### Lo que aprendiste hoy

| Concepto | Función |
|---|---|
| Cruzar dos variables | `table(x, y)` |
| Porcentajes por fila | `prop.table(tabla, margin = 1)` |
| Un promedio por grupo | `tapply(numero, grupo, mean)` |
| Comparar grupos con un gráfico | `boxplot(y ~ grupo)` |
| Pasar todo a minúscula | `tolower()` *(nueva)* |
| Sacar espacios sobrantes | `trimws()` *(nueva)* |

### Las cuatro ideas que hay que llevarse

1. **Que el código corra no significa que el resultado sirva.** Una persona que escribe `"3 horas"`
   deja en `NA` el promedio de las 33.
2. **Un porcentaje sin su `n` puede mentir.** El "100%" del metro son 6 personas; el de la categoría
   `Otro`, una sola.
3. **Limpiar es por capas, y cada capa es una decisión que se documenta.** Mayúsculas, espacios,
   tildes.
4. **Cuando la base se acaba, se acabó.** No hay código que arregle tener sólo 33 casos. Hay que
   salir a buscar datos de verdad.

### Siguiente paso

En la segunda mitad de la clase vamos a mirar juntos una base con **96.122 personas y 32 años de
historia** — la Encuesta CEP — y después van a recibir el **catálogo de fuentes** para elegir de
dónde van a salir los datos de su proyecto.